In [ ]:
import datetime
import sys
from pathlib import Path
from collections import Counter
import re
import numpy as np
from PIL import Image
import pytesseract
import cv2


# =========================
# CONFIG
# =========================

# On your Windows machine, adjust this if needed:
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# OCR / ADA thresholds
OCR_CONF_THRESHOLD = 60             # minimum OCR confidence for a text box

# Text size thresholds
MIN_TEXT_HEIGHT_PX_DIGITAL = 14     # digital legibility threshold (px)
PRINT_WIDTH_INCH = 4.5              # assumed printed width (strict)
MIN_LABEL_PT_PRINT = 9.0            # minimum label font size (pt)
TEXT_HEIGHT_TO_PT_SCALE = 3.0       # fudge factor px -> pt

# Contrast thresholds
MIN_CONTRAST_TEXT = 4.5             # WCAG AA small text
MIN_CONTRAST_GRAPHIC = 3.0          # non-text graphics

# Blur / DPI thresholds
BLUR_THRESHOLD = 80.0               # variance of Laplacian; lower = blurrier
MIN_PRINT_DPI = 300                 # strict print DPI requirement

# Color-blind / grayscale
COLOR_CLUSTER_K = 6                 # how many color clusters to consider
DELTAE_COLLISION_THRESHOLD = 15.0   # ΔE*ab below this = colors effectively indistinguishable


# =========================
# WCAG CONTRAST UTILITIES
# =========================

def srgb_to_linear(c):
    if c <= 0.04045:
        return c / 12.92
    return ((c + 0.055) / 1.055) ** 2.4


def relative_luminance(rgb):
    r, g, b = [x / 255.0 for x in rgb]
    r_lin = srgb_to_linear(r)
    g_lin = srgb_to_linear(g)
    b_lin = srgb_to_linear(b)
    return 0.2126 * r_lin + 0.7152 * g_lin + 0.0722 * b_lin


def contrast_ratio(c1, c2):
    L1 = relative_luminance(c1)
    L2 = relative_luminance(c2)
    L_light = max(L1, L2)
    L_dark = min(L1, L2)
    return (L_light + 0.05) / (L_dark + 0.05)


# =========================
# TEXT CONTRAST & SIZE
# =========================

def analyze_text_contrast_and_size(img_np):
    """
    Strict text analysis:
      - Use OCR to find words with conf >= OCR_CONF_THRESHOLD
      - Sample foreground (center of word box) & background (2px above)
      - Track worst contrast and smallest text height
      - If OCR finds no text → all text checks FAIL so you can manually review.
    """
    img_pil = Image.fromarray(img_np)
    ocr = pytesseract.image_to_data(img_pil, output_type=pytesseract.Output.DICT)

    has_text = False
    min_contrast = None
    min_text_height = None

    n = len(ocr["text"])
    h_img, w_img, _ = img_np.shape

    for i in range(n):
        text = ocr["text"][i].strip()
        try:
            conf = int(ocr["conf"][i])
        except ValueError:
            continue

        if not text:
            continue
        if conf < OCR_CONF_THRESHOLD:
            continue

        has_text = True
        x = ocr["left"][i]
        y = ocr["top"][i]
        w_box = ocr["width"][i]
        h_box = ocr["height"][i]

        if h_box <= 0 or w_box <= 0:
            continue

        if (min_text_height is None) or (h_box < min_text_height):
            min_text_height = h_box

        cx = min(x + w_box // 2, w_img - 1)
        cy = min(y + h_box // 2, h_img - 1)

        fg = img_np[cy, cx, :]

        by = max(y - 2, 0)
        bx = cx
        bg = img_np[by, bx, :]

        cr = contrast_ratio(fg, bg)
        if (min_contrast is None) or (cr < min_contrast):
            min_contrast = cr

    # If OCR finds no text, mark text checks as FAIL
    if not has_text:
        return {
            "has_text": False,
            "min_contrast": None,
            "contrast_ok": False,
            "min_text_height_px": None,
            "digital_text_size_ok": False,
            "print_text_size_ok": False,
            "estimated_min_point_size": None,
        }

    # Digital text size
    digital_text_size_ok = (min_text_height >= MIN_TEXT_HEIGHT_PX_DIGITAL)

    # Print size: approximate DPI from width
    img_h, img_w, _ = img_np.shape
    dpi = img_w / PRINT_WIDTH_INCH
    est_pt = (min_text_height / dpi) * 72.0 * TEXT_HEIGHT_TO_PT_SCALE
    print_text_size_ok = (est_pt >= MIN_LABEL_PT_PRINT)

    contrast_ok = (min_contrast is not None) and (min_contrast >= MIN_CONTRAST_TEXT)

    return {
        "has_text": True,
        "min_contrast": float(min_contrast) if min_contrast is not None else None,
        "contrast_ok": bool(contrast_ok),
        "min_text_height_px": int(min_text_height) if min_text_height is not None else None,
        "digital_text_size_ok": bool(digital_text_size_ok),
        "print_text_size_ok": bool(print_text_size_ok),
        "estimated_min_point_size": float(est_pt) if est_pt is not None else None,
    }


# =========================
# BLUR / DPI
# =========================

def analyze_blur(img_np):
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    blur_metric = lap.var()
    blur_ok = blur_metric >= BLUR_THRESHOLD
    return float(blur_metric), bool(blur_ok)


def estimate_dpi(img_np):
    h, w, _ = img_np.shape
    dpi = w / PRINT_WIDTH_INCH
    return float(dpi), bool(dpi >= MIN_PRINT_DPI)


# =========================
# GRAPHICS CONTRAST (BARS / AREAS)
# =========================

def rgb_to_hsv_np(rgb_arr):
    rgb = rgb_arr.astype(np.float32) / 255.0
    return cv2.cvtColor(rgb.reshape(1, -1, 3), cv2.COLOR_RGB2HSV)[0]


def estimate_background_color(img_np):
    """
    Estimate background as very light, low-saturation pixels.
    Fallback: lightest 10% of pixels by V.
    """
    h, w, _ = img_np.shape
    flat = img_np.reshape(-1, 3).astype(np.float32)
    hsv = rgb_to_hsv_np(flat)
    H, S, V = hsv[:, 0], hsv[:, 1], hsv[:, 2]

    bg_mask = (V > 0.92) & (S < 0.10)
    if np.count_nonzero(bg_mask) > 500:
        bg_pixels = flat[bg_mask]
    else:
        idx = np.argsort(V)
        top_k = idx[int(len(idx) * 0.9):]
        bg_pixels = flat[top_k]

    bg_rgb = np.median(bg_pixels, axis=0)
    return bg_rgb


def segment_graphic_objects(img_np, bg_rgb, diff_threshold=8, min_area=200, min_width=20, min_height=5):
    """
    Roughly segment non-text graphic objects (bars, slices, etc.)
    by difference from background.
    """
    diff = np.linalg.norm(img_np.astype(float) - bg_rgb[None, None, :], axis=2)
    mask = diff > diff_threshold

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8
    )

    objects = []
    for i in range(1, num_labels):  # 0 = background
        x, y, w, h, area = stats[i]
        if area < min_area:
            continue
        if w < min_width and h < min_height:
            continue
        if h < 4 or w < 4:
            continue
        objects.append((x, y, w, h))

    return objects


def dominant_color_for_object(img_np, bbox):
    """
    Within an object bounding box, remove near-white pixels and
    cluster remaining colors; choose the largest cluster as the dominant color.
    """
    x, y, w, h = bbox
    region = img_np[y:y+h, x:x+w].reshape(-1, 3).astype(np.float32)

    hsv = rgb_to_hsv_np(region)
    H, S, V = hsv[:, 0], hsv[:, 1], hsv[:, 2]
    bg_like = (V > 0.92) & (S < 0.10)
    fg_pixels = region[~bg_like]

    if len(fg_pixels) < 50:
        return None, False

    if len(fg_pixels) > 5000:
        idx = np.random.choice(len(fg_pixels), 5000, replace=False)
        sample = fg_pixels[idx]
    else:
        sample = fg_pixels

    Z = sample.astype(np.float32)
    K = 3
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(
        Z, K, None, criteria, 5, cv2.KMEANS_RANDOM_CENTERS
    )

    labels = labels.flatten()
    counts = np.bincount(labels, minlength=K)
    dom_idx = np.argmax(counts)
    dom_rgb = centers[dom_idx]
    return dom_rgb, True


def analyze_graphics_contrast_for_image_array(img_np):
    """
    Non-text graphic objects contrast:
      - segment against background
      - compute dominant color for each object
      - compare to background with MIN_CONTRAST_GRAPHIC
    """
    bg_rgb = estimate_background_color(img_np)
    bg_lum = relative_luminance(bg_rgb)

    objects_bboxes = segment_graphic_objects(img_np, bg_rgb)

    objects = []
    min_contrast = None
    fail_count = 0

    for idx, bbox in enumerate(objects_bboxes, start=1):
        dom_rgb, ok = dominant_color_for_object(img_np, bbox)
        if not ok:
            objects.append({
                "object_id": idx,
                "bbox": bbox,
                "dominant_rgb": None,
                "contrast": None,
                "passes_contrast": False,
                "reason": "Insufficient non-background pixels",
            })
            fail_count += 1
            continue

        cr = contrast_ratio(dom_rgb, bg_rgb)
        passes = cr >= MIN_CONTRAST_GRAPHIC

        if (min_contrast is None) or (cr < min_contrast):
            min_contrast = cr
        if not passes:
            fail_count += 1

        objects.append({
            "object_id": idx,
            "bbox": bbox,
            "dominant_rgb": [float(dom_rgb[0]), float(dom_rgb[1]), float(dom_rgb[2])],
            "contrast": float(cr),
            "passes_contrast": bool(passes),
            "reason": "OK" if passes else f"Contrast {cr:.2f} < {MIN_CONTRAST_GRAPHIC:.1f}",
        })

    graphics_ada_pass = (fail_count == 0)

    summary = {
        "background_rgb": [float(bg_rgb[0]), float(bg_rgb[1]), float(bg_rgb[2])],
        "background_luminance": float(bg_lum),
        "num_graphic_objects": len(objects_bboxes),
        "num_graphic_fail": fail_count,
        "min_graphic_contrast": float(min_contrast) if min_contrast is not None else None,
        "graphics_ada_pass": bool(graphics_ada_pass),
        "objects": objects,
    }

    return summary


# =========================
# COLOR-BLIND COLLISION TEST (PAIRWISE)
# =========================

def simulate_colorblind_rgb(rgb_arr, matrix):
    """
    Apply a 3x3 simulation matrix to RGB colors.
    rgb_arr: (N,3) float or uint8 in 0–255.
    """
    flat = rgb_arr.reshape(-1, 3).astype(np.float32)
    transformed = flat.dot(matrix.T)
    transformed = np.clip(transformed, 0, 255)
    return transformed.reshape(rgb_arr.shape).astype(np.uint8)


# Approximate matrices for CVD simulations
DEUTERANOPIA = np.array([[0.367, 0.861, -0.228],
                         [0.280, 0.673,  0.047],
                         [-0.012, 0.043, 0.968]])

PROTANOPIA = np.array([[0.152, 0.829, 0.019],
                       [0.115, 0.878, 0.007],
                       [-0.004, 0.034, 0.970]])

TRITANOPIA = np.array([[0.960, 0.040, 0.000],
                       [0.000, 0.818, 0.182],
                       [0.000, 0.258, 0.742]])


def rgb_to_lab(rgb_color):
    """
    Convert a single RGB color (3,) in 0–255 to Lab.
    """
    rgb = np.array(rgb_color, dtype=np.uint8).reshape(1, 1, 3)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    return lab[0, 0, :].astype(float)


def deltaE_lab(c1_lab, c2_lab):
    """
    Simple CIE76 ΔE in Lab space.
    """
    return float(np.linalg.norm(c1_lab - c2_lab))


def extract_dominant_colors(img_np, top_n=COLOR_CLUSTER_K):
    """
    Extract overall dominant colors using K-means in RGB space.
    Returns list of cluster dicts: {id, rgb, hex, luminance, count}
    """
    # Downsample for speed
    h, w, _ = img_np.shape
    target_w = 300
    scale = target_w / float(w) if w > target_w else 1.0
    new_w = int(w * scale)
    new_h = int(h * scale)
    if scale != 1.0:
        small = cv2.resize(img_np, (new_w, new_h), interpolation=cv2.INTER_AREA)
    else:
        small = img_np.copy()

    pixels = small.reshape(-1, 3).astype(np.float32)
    if pixels.shape[0] > 20000:
        idx = np.random.choice(pixels.shape[0], 20000, replace=False)
        pixels = pixels[idx]

    K = min(top_n, pixels.shape[0])
    if K < 1:
        return []

    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(
        pixels, K, None, criteria, 5, cv2.KMEANS_RANDOM_CENTERS
    )

    labels = labels.flatten()
    counts = Counter(labels)
    clusters = []
    for idx, count in counts.most_common(K):
        rgb = centers[idx]
        r, g, b = int(rgb[0]), int(rgb[1]), int(rgb[2])
        hex_color = "#{:02X}{:02X}{:02X}".format(r, g, b)
        lum = relative_luminance((r, g, b))
        clusters.append({
            "id": len(clusters) + 1,
            "rgb": (r, g, b),
            "hex": hex_color,
            "luminance": float(lum),
            "count": int(count),
        })

    return clusters


def analyze_colorblind_collisions(img_np):
    """
    Use K-means clusters as "palette colors", then:
      - Convert each cluster center to Lab
      - Simulate each cluster center under Deuteranopia, Protanopia, Tritanopia
      - Compute ΔE between every pair of simulated colors
      - If ΔE < threshold, mark as collision
    Returns:
      {
        "clusters": [...],
        "collisions": {
            "deuteranopia": [(id1, id2, deltaE), ...],
            "protanopia":   [...],
            "tritanopia":   [...],
        },
        "has_colorblind_collision": bool
      }
    """
    clusters = extract_dominant_colors(img_np, top_n=COLOR_CLUSTER_K)
    if len(clusters) <= 1:
        return {
            "clusters": clusters,
            "collisions": {
                "deuteranopia": [],
                "protanopia": [],
                "tritanopia": [],
            },
            "has_colorblind_collision": False,
        }

    # Original Lab colors for each cluster
    lab_orig = [rgb_to_lab(c["rgb"]) for c in clusters]

    # Prepare RGB centers as array
    rgb_centers = np.array([c["rgb"] for c in clusters], dtype=np.uint8)

    # Simulate each mode
    sims = {
        "deuteranopia": simulate_colorblind_rgb(rgb_centers, DEUTERANOPIA),
        "protanopia":   simulate_colorblind_rgb(rgb_centers, PROTANOPIA),
        "tritanopia":   simulate_colorblind_rgb(rgb_centers, TRITANOPIA),
    }

    collisions = {"deuteranopia": [], "protanopia": [], "tritanopia": []}

    # For each simulation, convert to Lab and compare pairwise ΔE
    for mode, sim_rgb in sims.items():
        sim_lab = [rgb_to_lab(rgb) for rgb in sim_rgb]

        n = len(sim_lab)
        for i in range(n):
            for j in range(i + 1, n):
                dE = deltaE_lab(sim_lab[i], sim_lab[j])
                if dE < DELTAE_COLLISION_THRESHOLD:
                    collisions[mode].append((clusters[i]["id"], clusters[j]["id"], float(dE)))

    has_collision = any(collisions[m] for m in collisions)

    return {
        "clusters": clusters,
        "collisions": collisions,
        "has_colorblind_collision": bool(has_collision),
    }


# =========================
# MAIN ANALYSIS FUNCTION
# =========================

def analyze_image_for_ada(image_path):
    img_path = Path(image_path)
    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)

    # 1) Text (contrast & size)
    text = analyze_text_contrast_and_size(img_np)

    # 2) Blur
    blur_metric, blur_ok = analyze_blur(img_np)

    # 3) DPI
    approx_dpi, dpi_ok = estimate_dpi(img_np)

    # 4) Graphics contrast (bars/areas)
    graphics = analyze_graphics_contrast_for_image_array(img_np)

    # 5) Color-blind collisions
    cb = analyze_colorblind_collisions(img_np)

    # Aggregate pass/fail (strict)
    digital_ada_pass = (
        text["contrast_ok"] and
        text["digital_text_size_ok"] and
        blur_ok and
        not cb["has_colorblind_collision"]
    )

    print_ada_pass = (
        text["contrast_ok"] and
        text["print_text_size_ok"] and
        blur_ok and
        dpi_ok and
        graphics["graphics_ada_pass"] and
        not cb["has_colorblind_collision"]
    )

    return {
        "image_file": img_path.name,
        "digital_ada_pass": bool(digital_ada_pass),
        "print_ada_pass": bool(print_ada_pass),

        "text": text,
        "blur_metric": blur_metric,
        "blur_ok": blur_ok,
        "approx_dpi": approx_dpi,
        "dpi_ok": dpi_ok,

        "graphics": graphics,

        "colorblind": cb,
    }


# =========================
# PRETTY PRINT REPORT
# =========================

def print_report(result):
    

    print(f"\n=== ADA REPORT for {result['image_file']} ===\n")

    # Simple summary
    print("SUMMARY")
    print("-------")
    print(f"  PRINT ADA PASS:     {result['print_ada_pass']}")
    print(f"  DIGITAL ADA PASS:   {result['digital_ada_pass']}")
    print(f"  TEXT OK:            {result['text']['contrast_ok'] and (result['text']['digital_text_size_ok'] or result['text']['print_text_size_ok'])}")
    print(f"  GRAPHICS OK:        {result['graphics']['graphics_ada_pass']}")
    print(f"  COLORBLIND SAFE:    {not result['colorblind']['has_colorblind_collision']}")
    print(f"  BLUR OK:            {result['blur_ok']}")
    print(f"  DPI OK:             {result['dpi_ok']}")
    print()


    # Text
    t = result["text"]
    print("TEXT / OCR")
    print("-----------")
    print(f"  Has text:                 {t['has_text']}")
    print(f"  Min text contrast:        {t['min_contrast']}")
    print(f"  Contrast OK (>=4.5:1):    {t['contrast_ok']}")
    print(f"  Min text height (px):     {t['min_text_height_px']}")
    print(f"  Est min point size:       {t['estimated_min_point_size']}")
    print(f"  Digital text size OK:     {t['digital_text_size_ok']}")
    print(f"  Print text size OK:       {t['print_text_size_ok']}")
    print()

    # Blur / DPI
    print("BLUR / DPI")
    print("-----------")
    print(f"  Blur metric:              {result['blur_metric']:.1f}  (ok={result['blur_ok']})")
    print(f"  Approx DPI (width):       {result['approx_dpi']:.1f}  (ok={result['dpi_ok']})")
    print()

    # Graphics
    g = result["graphics"]
    print("GRAPHICS (Bars/Areas)")
    print("---------------------")
    print(f"  Num graphic objects:      {g['num_graphic_objects']}")
    print(f"  Num graphic fails:        {g['num_graphic_fail']}")
    print(f"  Min graphic contrast:     {g['min_graphic_contrast']}")
    print(f"  Graphics ADA PASS:        {g['graphics_ada_pass']}")
    print()

    # Colorblind
    cb = result["colorblind"]
    print("COLOR-BLINDNESS (Pairwise Cluster Collisions)")
        # --- Per-color PASS/FAIL summary ---
    print("  Per-Color Status:")
    color_fail_map = {c['id']: False for c in cb["clusters"]}

    # Mark failures based on collisions
    for mode in ["deuteranopia", "protanopia", "tritanopia"]:
        for (id1, id2, dE) in cb["collisions"][mode]:
            color_fail_map[id1] = True
            color_fail_map[id2] = True

    for c in cb["clusters"]:
        cid = c["id"]
        if color_fail_map[cid]:
            print(f"    Cluster {cid} ({c['hex']}): FAIL (collides under color blindness)")
        else:
            print(f"    Cluster {cid} ({c['hex']}): PASS")
    print()

    # print("--------------------------------------------")
    # print(f"  Has colorblind collision: {cb['has_colorblind_collision']}")
    # print()
    # print("  Dominant color clusters:")
    # for c in cb["clusters"]:
    #     print(f"    Cluster {c['id']}: {c['hex']}  RGB{c['rgb']}  lum={c['luminance']:.3f}  count={c['count']}")
    # print()

    for mode in ["deuteranopia", "protanopia", "tritanopia"]:
        col_list = cb["collisions"][mode]
        print(f"  Collisions under {mode}:")
        if not col_list:
            print("    (none)")
        else:
            for (id1, id2, dE) in col_list:
                print(f"    Cluster {id1} <-> Cluster {id2}  ΔE={dE:.2f}  (FAIL)")
        print()

    print("NOTES:")
    print("  - Any OCR failure (no text found) causes text checks to fail so you can visually review.")
    print("  - Any color pair with ΔE < "
          f"{DELTAE_COLLISION_THRESHOLD:.1f} under a CVD simulation is treated as a collision.")
    print("  - This is intentionally conservative for a Congressional-level ADA review.")
    print()

def append_report_to_html(html_path, image_path, report_text, result):
    """
    Appends an ADA report section to an HTML file.
    - html_path: Path to .html file
    - image_path: Path to the analyzed image
    - report_text: The printed report as a string
    """
    from html import escape
    html_path = Path(html_path)
    print("html path:", html_path)
    # Create file with header IF it doesn't exist
    if not html_path.exists():
        with open(html_path, "w", encoding="utf-8") as f:
            f.write("<html><head><meta charset='utf-8'>")
            f.write("<title>ADA Image Report</title>")
            f.write("<style>body{font-family:Arial;} pre{background:#f0f0f0;padding:10px;} img{max-width:600px;border:1px solid #999;margin-bottom:10px;} h2{border-bottom:1px solid #ccc;}</style>")
            f.write("</head><body>\n")
            f.write("<h1>ADA Compliance Report</h1>\n")

    # Append the new report block
    with open(html_path, "a", encoding="utf-8") as f:
    #    f.write("<hr>\n")
      
        f.write(f"<h2>{escape(Path(image_path).name)}</h2>\n")
        f.write(f"<img src='{escape(str(image_path))}' width='250px' height='250px' alt='Image preview'>\n")
        f.write(f"<p>Date: {str(datetime.datetime.now())}</p>\n")

        f.write("<pre>")
        f.write(escape(report_text))
        f.write("</pre>\n")

        cb = result["colorblind"]
        f.write("<p>COLOR-BLINDNESS (Pairwise Cluster Collisions)</p>\n")
            # --- Per-color PASS/FAIL summary ---
        f.write("<br>  Per-Color Status:<br>")
        color_fail_map = {c['id']: False for c in cb["clusters"]}

        # Mark failures based on collisions
        for mode in ["deuteranopia", "protanopia", "tritanopia"]:
            for (id1, id2, dE) in cb["collisions"][mode]:
                color_fail_map[id1] = True
                color_fail_map[id2] = True

        for c in cb["clusters"]:
            cid = c["id"]
            if color_fail_map[cid]:
                f.write(f"    Cluster {cid} <div style='display:inline-block;width:40px;height:20px; border:1px solid #000;margin:4px;background:{c['hex']};'></div><span>{c['hex']}</span>: FAIL (collides under color blindness)<br>")
            else:
                f.write(f"    Cluster {cid} <div style='display:inline-block;width:40px;height:20px; border:1px solid #000;margin:4px;background:{c['hex']};'></div><span>{c['hex']}</span>: PASS<br>")
        

        # hex_colors = re.findall(r"#[0-9A-Fa-f]{6}", report_text)

        # if hex_colors:
        #     f.write("<div><b>Color Swatches:</b><br>")
        #     for h in hex_colors:
        #         f.write(
        #             f"<div style='display:inline-block;width:40px;height:20px; border:1px solid #000;margin:4px;background:{h};'></div><span>{h}</span><br>"
        #         )
        #     f.write("</div>\n")


import io
import sys

def run_and_export(image_path, html_path="ada_summary.html"):
    """
    Runs the existing ADA analysis (unchanged)
    and appends the printed report + image to the HTML summary.
    """
    buffer = io.StringIO()
    real_stdout = sys.stdout
    sys.stdout = buffer

    # Your existing analysis functions – unchanged
    result = analyze_image_for_ada(image_path)
    print_report(result)

    sys.stdout = real_stdout
    report_text = buffer.getvalue()

    append_report_to_html(html_path, image_path, report_text,result)

    print(f"✔ Added {image_path} to {html_path}")
    return result


# =========================
# CLI ENTRY
# =========================

if __name__ == "__main__":
    # if len(sys.argv) < 2:
    #     print("Usage: python ada_single_image_full.py path\\to\\image.png")
    #     sys.exit(1)

    image_path = r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\Final/"
    img = input("Enter image dir and filename (with extension): ")
    image_path += img
     # sys.argv[1]
    # res = analyze_image_for_ada(image_path)
    # print_report(res)

    run_and_export(image_path)

html path: ada_summary.html
✔ Added \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\ada_output\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\excel_images\F 2-16_chart_1.png to ada_summary.html
